In [1]:
import sys
from pathlib import Path

# Get parent directory (Thesis-Edvin)
sys.path.append(str(Path.cwd().parent))

In [2]:
from utils import *
from datasets import load_dataset
from rag import *

In [3]:
gpt_pred = load_dataset("Eathus/github-issues-vul-detection-gpt-few-results-final-best2", split="test")
gpt_pred_df = gpt_pred.to_pandas()
gpt_pred_df = gpt_pred_df[~gpt_pred_df.duplicated(subset="issue_github_id", keep=False)]

In [4]:
xgb_pred = load_dataset("Eathus/github-issues-vul-detection-xgb-results", split='test')
xgb_pred_df = xgb_pred.to_pandas()
xgb_pred_df = xgb_pred_df[~xgb_pred_df.duplicated(subset="issue_github_id", keep=False)]

In [5]:
cip_pipeline_pred = pd.merge(gpt_pred_df, xgb_pred_df[['issue_github_id', 'xgb_prediction']], on='issue_github_id')
cip_pipeline_pred["cp_tf_label"] = cip_pipeline_pred.xgb_prediction & cip_pipeline_pred.gpt_is_relevant

In [41]:
true_pos = cip_pipeline_pred[cip_pipeline_pred.cp_tf_label & ~cip_pipeline_pred.cve_id.isna()]
false_pos = cip_pipeline_pred[cip_pipeline_pred.cp_tf_label & cip_pipeline_pred.cve_id.isna()]
pos = cip_pipeline_pred[cip_pipeline_pred.cp_tf_label]
false_neg = cip_pipeline_pred[~cip_pipeline_pred.cp_tf_label & ~cip_pipeline_pred.cve_id.isna()]
all_true = cip_pipeline_pred[~cip_pipeline_pred.cve_id.isna()]

print("true pos:", len(true_pos))
print("false pos:", len(false_pos))
print("pos:", len(pos))
print("false neg:", len(false_neg))
print("all true:", len(all_true))
print("all", len(cip_pipeline_pred))

true pos: 250
false pos: 137
pos: 387
false neg: 56
all true: 306
all 1763


In [7]:
gpt_pred_large = load_dataset("Eathus/github-issues-vul-max-detection-gpt-few-results-final3", split="test")
gpt_pred_large_df = gpt_pred_large.to_pandas()
gpt_pred_large_df = gpt_pred_large_df[~gpt_pred_large_df.duplicated(subset="issue_github_id", keep=False)]

In [8]:
xgb_pred_large = load_dataset("Eathus/github-issues-vul-detection-max-xgb-results", split='test')
xgb_pred_large_df = xgb_pred_large.to_pandas()
xgb_pred_large_df = xgb_pred_large_df[~xgb_pred_large_df.duplicated(subset="issue_github_id", keep=False)]

In [9]:
cip_pipeline_pred_large = pd.merge(gpt_pred_large_df, xgb_pred_large_df[['issue_github_id', 'xgb_prediction']], on='issue_github_id')
cip_pipeline_pred_large["cp_tf_label"] = cip_pipeline_pred_large.xgb_prediction & cip_pipeline_pred_large.gpt_is_relevant

In [42]:
true_pos_large = cip_pipeline_pred_large[cip_pipeline_pred_large.cp_tf_label & ~cip_pipeline_pred_large.cve_id.isna()]
false_pos_large = cip_pipeline_pred_large[cip_pipeline_pred_large.cp_tf_label & cip_pipeline_pred_large.cve_id.isna()]
pos_large = cip_pipeline_pred_large[cip_pipeline_pred_large.cp_tf_label]
false_neg_large = cip_pipeline_pred_large[~cip_pipeline_pred_large.cp_tf_label & ~cip_pipeline_pred_large.cve_id.isna()]
all_true_large = cip_pipeline_pred_large[~cip_pipeline_pred_large.cve_id.isna()]

print("true pos:", len(true_pos_large))
print("false pos:", len(false_pos_large))
print("pos:", len(pos_large))
print("false neg:", len(false_neg_large))
print("all true:", len(all_true_large))
print("all", len(cip_pipeline_pred_large))

true pos: 1204
false pos: 257
pos: 1461
false neg: 160
all true: 1364
all 6419


In [11]:
cip_pipeline_pred_large.cve_primary_cwe.value_counts()

cve_primary_cwe
No_CWE    5055
79         239
787        142
476        101
125         67
          ... 
459          1
749          1
521          1
1236         1
407          1
Name: count, Length: 115, dtype: int64

In [12]:
cip_pipeline_pred_large.cp_tf_label.value_counts()

cp_tf_label
False    4958
True     1461
Name: count, dtype: int64

In [13]:
cwes = load_dataset("Eathus/cwe_view1000_list_gpt_few_cwe_desc_final_best", split="train")
cwes_df = cwes.to_pandas()

In [52]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class ReplySchema(BaseModel):
    gpt_cwe: str = Field(
        description="The CWE-ID (*only the number*) of the CWE entry that best fits the vulnerability description"
    )
    gpt_cwe_confidence: int = Field(
        description="An integer from 1 to 5 indicating your level of confidence  (1 = very low, 2 = low, 3 = medium, 4 = high, 5 = very high)."
    )


llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    api_key=OPENAI_API_KEY_KTH,  # <- this overrides the default
)  # maybe set max_token to 14000

prompts_dict = load_prompts(os.getcwd() + "/../utils/prompts")

prompt = ChatPromptTemplate.from_messages(
    [("system", prompts_dict["RAG_system_setup_json"]), ("human", "{desc}")],
)


def parser(message: ReplySchema):
    return message.model_dump_json()


llm = llm.with_structured_output(ReplySchema)
chain = prompt | llm | parser

In [15]:
from pandarallel import pandarallel
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
)
from openai import OpenAIError, RateLimitError  # Explicitly import errors

pandarallel.initialize(progress_bar=True, nb_workers=15)

@retry(
    stop=stop_after_attempt(5),  # Retry up to 5 times
    wait=wait_exponential(multiplier=2, min=1, max=60),  # Exponential backoff
    retry=retry_if_exception_type(RateLimitError),  # Retry only on rate limit errors
)
def _gpt_classify(desc, cwe_entries):
    if (
        not desc
        or not isinstance(desc, str)
        or not cwe_entries
        or not isinstance(cwe_entries, str)
    ):  # Check for empty/invalid messages
        return None
    return chain.invoke(
        {
            "cwe_entries": cwe_entries,
            "desc": desc,
        }
    )  # Adjusted for OpenAI API format


def gpt_classify(desc, cwe_entries):
    try:
        return _gpt_classify(desc, cwe_entries)
    except OpenAIError as e:  # Catch all OpenAI-specific errors
        print(f"OpenAI API error: {e}")
    except Exception as e:
        print(f"General error processing message: {e}")
    return None

INFO: Pandarallel will run on 15 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [16]:
cwes_df.columns

Index(['ID', 'Name', 'Abstraction', 'Structure', 'Status', 'Description',
       'ExtendedDescription', 'ApplicablePlatforms', 'AlternateTerms',
       'ModesOfIntroduction', 'CommonConsequences', 'PotentialMitigations',
       'ObservedExamples', 'AffectedResources', 'TaxonomyMappings',
       'RelatedAttackPatterns', 'References', 'Notes', 'ContentHistory',
       'MappingNotes_Usage', 'MappingNotes_Rationale', 'MappingNotes_Comments',
       'MappingNotes_Reasons', 'MappingNotes_Suggestions', 'RelatedWeaknesses',
       'WeaknessOrdinalities', 'DetectionMethods', 'DemonstrativeExamples',
       'FunctionalAreas', 'Diagram', 'LikelihoodOfExploit',
       'BackgroundDetails', 'NumPaths', 'Paths', 'Children', 'Summary',
       'gpt_cwe_description'],
      dtype='object')

In [32]:
def bare_rag_label(
    data_df, vectorstore, match_col="gpt_vulnerability", k=1
):  # max for k is 20
    ret_df = data_df.copy()
    faiss_retriever = vectorstore.as_retriever(
        search_kwargs={
            "k": k,
        }
    )

    #bm25_retriever = BM25Retriever.from_documents(bm25_docs)
    #bm25_retriever.k = k
    #ensemble_retriever = EnsembleRetriever(
    #    retrievers=[bm25_retriever, faiss_retriever],
    #    weights=[0.5, 0.5],
    #    # retrievers=[faiss_retriever], weights=[1]
    #)

    # display(ret_df[ret_df['cve_primary_cwe'].isna()])
    def label(desc):
        try:
            ret = list(
                set([x.metadata["CWE_ID"] for x in faiss_retriever.invoke(desc)])
            )
            # print(len(ret))
            return ret
        except Exception as e:
            print(f"General error processing message: {e}")
            return

    ret_df["rag_candidates"] = [[] for _ in range(len(ret_df))]
    ret_df.loc[ret_df.cp_tf_label == True, "rag_candidates"] = ret_df.loc[ret_df.cp_tf_label == True, match_col].progress_map(label)
    ret_df["rag_label"] = ret_df.apply(
        lambda row: (
            row["cve_primary_cwe"]
            if row["cve_primary_cwe"] in row["rag_candidates"]
            else row["rag_candidates"][0] if row["rag_candidates"] != [] else 'No_CWE' 
        ),
        axis=1,
    )

    return ret_df

In [ ]:
import pickle
import time

def gpt_query(df, cwes_json,  desc_field='gpt_description', max_retries=10, retry_delay=1, overwrite=False, backup_file_path='tmp/rag_cwe_pred.pkl') :
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    dfc = df.copy()

    dfc = dfc.reset_index(drop=True)
    cwe_dict = {cwe['ID']: cwe for cwe in cwes_json["Weaknesses"]}
    # Check if the file exists
    if os.path.exists(backup_file_path) and not overwrite:
        with open(backup_file_path, "rb") as file:
            dfc = pickle.load(file)
        print("Pickle file loaded successfully!")
    else:
        print(f"The file at {backup_file_path} does not exist or is set to be overwritten. Setting rag_prediction to 'None'")
        dfc["rag_prediction"] = None
        dfc.loc[dfc.cp_tf_label == False, "rag_prediction"] = "{\"gpt_cwe\": \"No_CWE\", \"gpt_cwe_confidence\": 5}" 
    
    retries = 0
    
    while (
        not dfc[dfc.rag_prediction.isna()].empty
        and retries < max_retries
    ):
        # Get indices of rows needing processing
        na_indices = dfc[
            dfc.rag_prediction.isna()
        ].index

        if len(na_indices) == 0:
            break

        # Process ONLY those rows and assign directly to original DF
        dfc.loc[na_indices, "rag_prediction"] = dfc.loc[
            na_indices, [desc_field, "rag_candidates"]
        ].parallel_apply(
            lambda x: gpt_classify(
                x[desc_field], cwe_list_to_json_str(x["rag_candidates"], cwe_dict, doc_density=CWE_doc_density.HEAVY)
            ),
            axis=1
        )

        with open(backup_file_path, "wb") as file:  # 'wb' mode writes in binary format
            pickle.dump(dfc, file)
        retries += 1
        print(f"Retry {retries}: Processed {len(na_indices)} rows")
        time.sleep(retry_delay)

    dfc['rag_prediction'] = dfc['rag_prediction'].map(
        lambda x: json.loads(x) if pd.notna(x) else None
    )
    dfc[['rag_prediction', 'rag_confidence']] = dfc['rag_prediction'].apply(
        lambda x: pd.Series({"1":x['gpt_cwe'], "2":x['gpt_cwe_confidence']}) if pd.notna(x) else None
    )
    dfc['rag_prediction'] = dfc['rag_prediction'].map(
        lambda x: x if pd.notna(x) and x in cwe_dict else 'No_CWE'
    )

    return dfc

# View CWE-1000

In [19]:
cwes_dict = cwes_df.to_dict('records')
documents_light = [
    create_ordered_cwe_document(weakness, old=True) for weakness in cwes_dict
]
document_dict = {doc.metadata["CWE_ID"]: doc for doc in documents_light}
documents = [
    Document(
        page_content=cwe.gpt_cwe_description,
        metadata=document_dict[cwe.ID].metadata,  # Original metadata
    )
    for _, cwe in cwes_df.iterrows()
]

allowed_usage = ["Allowed", "Allowed-with-Review", "Discouraged"]
documents = filter_cwe_docs_usage(documents, allowed_usage)

documents = split_docs(documents, echo=True, max_tokens=800)

Original: 881 docs
After splitting: 901 docs


In [20]:
vectorstore = create_vectorstore(documents, "tmp/faiss_enhanced_kb_index")

Loading existing vector store...


In [21]:
with open("tmp/view_CWE-1000_all_weaknesses.json", "r") as file: #JSON file generated in CWE_analysis
    cwes_json = json.load(file)

In [23]:
cip_pipeline_pred.columns

Index(['cve_id', 'cve_published', 'cve_descriptions', 'cve_metrics',
       'cve_references', 'cve_configurations', 'cve_primary_cwe', 'cve_tags',
       'issue_owner_repo', 'issue_body', 'issue_title', 'issue_comments_url',
       'issue_comments_count', 'issue_created_at', 'issue_updated_at',
       'issue_html_url', 'issue_github_id', 'issue_number', 'label',
       'issue_msg', 'issue_msg_n_tokens', 'issue_embedding',
       '__index_level_0__', 'gpt_description', 'gpt_vulnerability',
       'gpt_confidence', 'gpt_is_relevant', 'xgb_prediction', 'cp_tf_label'],
      dtype='object')

## Small Test

### Full

In [43]:
ranges = [5, 10, 15, 20]
for k in ranges :
    rag_labeled_df = bare_rag_label(
        cip_pipeline_pred, vectorstore, match_col="gpt_vulnerability", k=k
    )
    rag_labeled_df = gpt_query(rag_labeled_df, cwes_json, backup_file_path=f'tmp/full_pipeline_cwe_pred_top-{k}.pkl', overwrite=True)
    evaluate_rag(rag_labeled_df, col='rag_prediction', save=True, md_filename=f'tmp/full_pipeline/classification_report_top-{k}.md')

 16%|█▋        | 63/387 [00:02<00:11, 27.66it/s]


KeyboardInterrupt: 

### Pos Only (FP + TP)

In [53]:
ranges = [5, 10, 15, 20]
for k in ranges :
    rag_labeled_df = bare_rag_label(
        pos, vectorstore, match_col="gpt_vulnerability", k=k
    )
    rag_labeled_df = gpt_query(rag_labeled_df, cwes_json, backup_file_path=f'tmp/full_pipeline_cwe_pred_top-{k}.pkl', overwrite=True)
    evaluate_rag(rag_labeled_df, col='rag_prediction', save=True, md_filename=f'tmp/full_pipeline/classification_report_top-{k}.md')

100%|██████████| 387/387 [00:13<00:00, 27.65it/s]


The file at tmp/full_pipeline_cwe_pred_top-5.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 387 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report_top-5.md


100%|██████████| 387/387 [00:13<00:00, 27.68it/s]


The file at tmp/full_pipeline_cwe_pred_top-10.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 387 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report_top-10.md


... (2 outputs left out)

Retry 1: Processed 387 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report_top-15.md


100%|██████████| 387/387 [00:13<00:00, 27.66it/s]


The file at tmp/full_pipeline_cwe_pred_top-20.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 387 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report_top-20.md


## Large Test

### Full

In [34]:
ranges = [5, 10, 15, 20]
for k in ranges :
    rag_labeled_df = bare_rag_label(
        cip_pipeline_pred_large, vectorstore, match_col="gpt_vulnerability", k=k
    )
    rag_labeled_df = gpt_query(rag_labeled_df, cwes_json, backup_file_path=f'tmp/large_full_pipeline_cwe_pred_top-{k}.pkl', overwrite=True)
    evaluate_rag(rag_labeled_df, col='rag_prediction', save=True, md_filename=f'tmp/full_pipeline/classification_report_large_top-{k}.md')


100%|██████████| 1461/1461 [00:54<00:00, 26.66it/s]


The file at tmp/large_full_pipeline_cwe_pred_top-5.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 1461 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report_large_top-5.md


100%|██████████| 1461/1461 [01:10<00:00, 20.76it/s]


The file at tmp/large_full_pipeline_cwe_pred_top-10.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 1461 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report_large_top-10.md


... (2 outputs left out)

Retry 1: Processed 1461 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report_large_top-15.md


100%|██████████| 1461/1461 [00:57<00:00, 25.44it/s]

The file at tmp/large_full_pipeline_cwe_pred_top-20.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 1461 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report_large_top-20.md


### Pos Only (FP + TP)

In [54]:
ranges = [5, 10, 15, 20]
for k in ranges :
    rag_labeled_df = bare_rag_label(
        pos_large, vectorstore, match_col="gpt_vulnerability", k=k
    )
    rag_labeled_df = gpt_query(rag_labeled_df, cwes_json, backup_file_path=f'tmp/large_full_pipeline_cwe_pred_top-{k}.pkl', overwrite=True)
    evaluate_rag(rag_labeled_df, col='rag_prediction', save=True, md_filename=f'tmp/full_pipeline/classification_report_large_top-{k}.md')


100%|██████████| 1461/1461 [00:55<00:00, 26.51it/s]

The file at tmp/large_full_pipeline_cwe_pred_top-5.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 1461 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report_large_top-5.md


100%|██████████| 1461/1461 [00:55<00:00, 26.45it/s]

The file at tmp/large_full_pipeline_cwe_pred_top-10.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 1461 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report_large_top-10.md


... (2 outputs left out)

Retry 1: Processed 1461 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report_large_top-15.md


100%|██████████| 1461/1461 [00:55<00:00, 26.50it/s]

The file at tmp/large_full_pipeline_cwe_pred_top-20.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 1461 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report_large_top-20.md


# View CWE-1003

## Small Test

In [35]:
view1003 = load_dataset("Eathus/cwe_view1003_raw_list", split="train")
view1003_df = view1003.to_pandas()

In [36]:
documents1003 = [doc for doc in documents if doc.metadata['CWE_ID'] in view1003_df['ID'].tolist()]

In [37]:
vectorstore1003 = create_vectorstore(documents1003, "tmp/faiss_enhanced_kb1003_index")

Loading existing vector store...


In [38]:
with open("tmp/view_CWE-1003_all_weaknesses.json", "r") as file: #JSON file generated in CWE_analysis
    cwes1003_json = json.load(file)

### Full

In [39]:
ranges = [3, 5, 10, 15, 20]
for k in ranges :
    rag_labeled_df = bare_rag_label(
        cip_pipeline_pred, vectorstore1003, match_col="gpt_vulnerability", k=k
    )
    rag_labeled_df = gpt_query(rag_labeled_df, cwes1003_json, backup_file_path=f'tmp/full_pipeline1003_cwe_pred_top-{k}.pkl')
    evaluate_rag(rag_labeled_df, col='rag_prediction', save=True, md_filename=f'tmp/full_pipeline/classification_report1003_top-{k}.md')


100%|██████████| 387/387 [00:14<00:00, 26.53it/s]

The file at tmp/full_pipeline1003_cwe_pred_top-3.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 387 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report1003_top-3.md


100%|██████████| 387/387 [00:14<00:00, 26.53it/s]

The file at tmp/full_pipeline1003_cwe_pred_top-5.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 387 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report1003_top-5.md


... (5 outputs left out)

Retry 1: Processed 387 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report1003_top-15.md


100%|██████████| 387/387 [00:14<00:00, 26.51it/s]

The file at tmp/full_pipeline1003_cwe_pred_top-20.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 387 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report1003_top-20.md


### Pos Only (FP + TP)

In [56]:
ranges = [3, 5, 10, 15, 20]
for k in ranges :
    rag_labeled_df = bare_rag_label(
        pos, vectorstore1003, match_col="gpt_vulnerability", k=k
    )
    rag_labeled_df = gpt_query(rag_labeled_df, cwes1003_json, backup_file_path=f'tmp/full_pipeline1003_cwe_pred_top-{k}.pkl', overwrite=True)
    evaluate_rag(rag_labeled_df, col='rag_prediction', save=True, md_filename=f'tmp/full_pipeline/classification_report1003_top-{k}.md')

100%|██████████| 387/387 [00:14<00:00, 27.46it/s]


The file at tmp/full_pipeline1003_cwe_pred_top-3.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 387 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report1003_top-3.md


100%|██████████| 387/387 [00:14<00:00, 27.44it/s]

The file at tmp/full_pipeline1003_cwe_pred_top-5.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 387 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report1003_top-5.md


... (5 outputs left out)

Retry 1: Processed 387 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report1003_top-15.md


100%|██████████| 387/387 [00:14<00:00, 27.29it/s]


The file at tmp/full_pipeline1003_cwe_pred_top-20.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 387 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report1003_top-20.md


## Large Test

In [40]:
ranges = [3, 5, 10, 15, 20]
for k in ranges :
    rag_labeled_df = bare_rag_label(
        cip_pipeline_pred_large, vectorstore1003, match_col="gpt_vulnerability", k=k
    )
    rag_labeled_df = gpt_query(rag_labeled_df, cwes1003_json, backup_file_path=f'tmp/large_full_pipeline1003_cwe_pred_top-{k}.pkl')
    evaluate_rag(rag_labeled_df, col='rag_prediction', save=True, md_filename=f'tmp/full_pipeline/classification_report1003_large_top-{k}.md')


100%|██████████| 1461/1461 [00:57<00:00, 25.54it/s]

The file at tmp/large_full_pipeline1003_cwe_pred_top-3.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 1461 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report1003_large_top-3.md


100%|██████████| 1461/1461 [00:57<00:00, 25.56it/s]

The file at tmp/large_full_pipeline1003_cwe_pred_top-5.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 1461 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report1003_large_top-5.md


... (5 outputs left out)

Retry 1: Processed 1461 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report1003_large_top-15.md


100%|██████████| 1461/1461 [00:54<00:00, 26.62it/s]


The file at tmp/large_full_pipeline1003_cwe_pred_top-20.pkl does not exist or is set to be overwritten. Setting rag_prediction to 'None'


Retry 1: Processed 1461 rows

✅ Markdown report saved to: tmp/full_pipeline/classification_report1003_large_top-20.md


### Pos Only (FP + TP)

In [ ]:
ranges = [3, 5, 10, 15, 20]
for k in ranges :
    rag_labeled_df = bare_rag_label(
        pos_large, vectorstore1003, match_col="gpt_vulnerability", k=k
    )
    rag_labeled_df = gpt_query(rag_labeled_df, cwes1003_json, backup_file_path=f'tmp/large_full_pipeline1003_cwe_pred_top-{k}.pkl')
    evaluate_rag(rag_labeled_df, col='rag_prediction', save=True, md_filename=f'tmp/full_pipeline/classification_report1003_large_top-{k}.md')